# 00 — Notebook environment check

Verifies that `sft/.venv-nb` contains the expected package versions, that the
tokenizer snapshot sha256 matches `sft/environment_manifest.json`, and that
the tokenizers library can load the snapshot offline.

**No dataset access.**

In [ ]:
# --- PARAMETERS ---
# Paths are resolved relative to this notebook's location (sft/notebooks/).
# REPO_ROOT is two levels up from sft/notebooks/.
import pathlib as _pl
_NB_DIR = _pl.Path(__file__).parent if '__file__' in dir() else _pl.Path().resolve()
_REPO_ROOT = _NB_DIR.parent.parent
MANIFEST_PATH = str(_REPO_ROOT / "sft" / "environment_manifest.json")
TOKENIZER_DIR = str(_REPO_ROOT / "sft" / "tokenizer_snapshot")
TOKENIZER_FILE = str(_REPO_ROOT / "sft" / "tokenizer_snapshot" / "tokenizer.json")
# Fixed ASCII probe string for tokenizer round-trip (no dataset content).
PROBE_STRING = "The patient presents with a mild fever and sore throat."

## Definitions

In [ ]:
import sys
import json
import hashlib
import importlib.metadata
import pathlib

def get_version(pkg: str) -> str:
    """Return the installed version of *pkg*, or 'NOT INSTALLED'."""
    try:
        return importlib.metadata.version(pkg)
    except importlib.metadata.PackageNotFoundError:
        return "NOT INSTALLED"

def sha256_file(path: str) -> str:
    """Return the hex SHA-256 digest of the file at *path*."""
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()

## In-notebook tests (must pass before any real data is read)

In [ ]:
# Test 1: manifest file exists
assert pathlib.Path(MANIFEST_PATH).exists(), f"Manifest not found: {MANIFEST_PATH}"

# Test 2: tokenizer file exists
assert pathlib.Path(TOKENIZER_FILE).exists(), f"tokenizer.json not found: {TOKENIZER_FILE}"

# Test 3: tokenizer directory exists
assert pathlib.Path(TOKENIZER_DIR).is_dir(), f"Tokenizer dir not found: {TOKENIZER_DIR}"

# Test 4: required packages are importable
for pkg in ["pandas", "transformers", "tokenizers", "datasketch", "langid", "ipykernel"]:
    assert get_version(pkg) != "NOT INSTALLED", f"Package not installed: {pkg}"

print("All pre-checks passed.")

## Python version

In [ ]:
print(f"Python version : {sys.version}")

## Package versions

In [ ]:
packages = [
    ("pandas",       "pandas"),
    ("transformers", "transformers"),
    ("tokenizers",   "tokenizers"),
    ("datasketch",   "datasketch"),
    ("langid",       "langid"),
    ("ipykernel",    "ipykernel"),
]

col_w = 16
print(f"{'Package':<{col_w}}  Version")
print("-" * (col_w + 12))
for label, pkg in packages:
    print(f"{label:<{col_w}}  {get_version(pkg)}")

## tokenizer.json SHA-256 and manifest assertion

In [ ]:
actual_sha = sha256_file(TOKENIZER_FILE)
print(f"tokenizer.json sha256: {actual_sha}")

with open(MANIFEST_PATH, encoding="utf-8") as fh:
    manifest = json.load(fh)

expected_sha = manifest["tokenizer_snapshot"]["files"]["tokenizer.json"]["sha256"]
assert actual_sha == expected_sha, (
    f"SHA-256 mismatch!\n  actual  : {actual_sha}\n  expected: {expected_sha}"
)
print("SHA-256 assertion passed.")

## Offline tokenizer load and probe encoding

In [ ]:
import os
# Prevent any network calls from the tokenizers library.
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"

from tokenizers import Tokenizer  # type: ignore

tok = Tokenizer.from_file(TOKENIZER_FILE)
encoding = tok.encode(PROBE_STRING)
print(f"Probe string token count: {len(encoding.ids)}")

## Status

In [ ]:
print("STATUS: OK — all checks passed, environment verified.")